# Vision Agent — Baseline Comparison (Week 4 → Week 5)

**Scope:** two segmentation models are compared on the 8-patient sanity sample
(`LUNG1-001, -013, -303, -378, -328, -380, -048, -280`). Per the agreed scope
reduction there is **no training from scratch and no fine-tuning of external
checkpoints**. UNet3D has been retired entirely and removed from the pipeline.

| Model | Source | Role |
|---|---|---|
| `nnunet` | nnU-Net v2, **self-configured and trained on our own NSCLC-Radiomics cohort** (`Dataset001_NSCLCRadiomics`, RTSTRUCT-derived GTV masks). Zero-mask fallback until the trained checkpoint is available. | Primary tumour (GTV) segmenter |
| `totalsegmentator` | TotalSegmentator v2 `lung_nodules` task (class 2 = lung nodule). | Lesion-focused comparison / anatomical context — see note below |

**On TotalSegmentator's role:** `lung_nodules` is trained for nodule *detection*,
not GTV delineation, so it may under-segment bulky tumours. Whether it stands as a
comparable segmenter or is repositioned as anatomical context (narrowing the search
region to lung tissue) is judged from the results here — not assumed up front.

Results are produced by `scripts/evaluation/run_baselines.py` and saved to
`results/segmentation/week5_baselines.csv`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path("../..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

## Load results

In [ ]:
results_path = PROJECT_ROOT / "results" / "segmentation" / "week5_baselines.csv"
results_df = pd.read_csv(results_path)
display(results_df)

summary = results_df.groupby("model").agg(
    dice_mean=("dice", "mean"), dice_std=("dice", "std"),
    hd95_mean=("hd95_mm", "mean"), hd95_std=("hd95_mm", "std"),
    vol_err_mean=("volume_error_cm3", "mean"), vol_err_std=("volume_error_cm3", "std"),
    time_mean=("inference_time_sec", "mean"), time_std=("inference_time_sec", "std"),
)
summary

## DSC distribution per model

In [ ]:
models = list(results_df["model"].unique())
dice_by_model = [results_df.loc[results_df["model"] == m, "dice"].values for m in models]

fig, ax = plt.subplots(figsize=(7, 5))
ax.boxplot(dice_by_model, tick_labels=models, showmeans=True)

rng = np.random.default_rng(42)
for i, values in enumerate(dice_by_model, start=1):
    jitter = rng.normal(0, 0.04, size=len(values))
    ax.scatter(np.full(len(values), i) + jitter, values, alpha=0.7, color="black", s=20, zorder=3)

ax.set_ylabel("Dice Similarity Coefficient (vs GTV)")
ax.set_title("DSC by model — 8 sanity-sample patients")
ax.set_ylim(-0.02, 1.0)
plt.tight_layout()
plt.show()

## Visual comparison for one patient (LUNG1-001)

Re-runs each model's inference for a single patient so we can overlay the predicted
masks on the CT. Red contours show the ground-truth GTV / predicted segmentation on
the axial slice with the largest GTV cross-section.

In [ ]:
from agents.vision.inference import (
    run_nnunet_inference,
    run_totalsegmentator_inference,
)

processed_dir = PROJECT_ROOT / "data" / "processed" / "radiomics"
nnunet_model_folder = (
    PROJECT_ROOT / "results" / "nnunet" / "Dataset001_NSCLCRadiomics"
    / "nnUNetTrainer_250epochs__nnUNetPlans__3d_fullres"
)
patient_id = "LUNG1-001"

ct = np.load(processed_dir / f"{patient_id}_ct.npy")
gt = np.load(processed_dir / f"{patient_id}_mask.npy")

nnunet_pred, _ = run_nnunet_inference(patient_id, processed_dir, nnunet_model_folder)
totalseg_pred, _ = run_totalsegmentator_inference(patient_id, processed_dir)

# nnU-Net's logger forces matplotlib's backend to "Agg" on import, which breaks
# inline figure display in the rest of this notebook - restore it.
%matplotlib inline

# Axial slice with the largest GTV cross-section
z = int(np.argmax(gt.sum(axis=(1, 2))))

panels = [
    ("CT", None),
    ("Ground truth (GTV)", gt),
    ("nnU-Net (own cohort)", nnunet_pred),
    ("TotalSegmentator (lung_nodules)", totalseg_pred),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
for ax, (title, mask) in zip(axes, panels):
    ax.imshow(ct[z], cmap="gray")
    if mask is not None and mask[z].sum() > 0:
        ax.contour(mask[z], levels=[0.5], colors="red", linewidths=1.5)
    ax.set_title(title)
    ax.axis("off")

fig.suptitle(f"{patient_id} — axial slice z={z}")
plt.tight_layout()
plt.show()

## Per-patient breakdown

In [ ]:
pivot = results_df.pivot(index="patient_id", columns="model", values="dice")

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(pivot.values, cmap="viridis", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)

for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        value = pivot.values[i, j]
        ax.text(j, i, f"{value:.3f}", ha="center", va="center",
                color="white" if value < 0.5 else "black")

ax.set_title("DSC vs GTV — per patient and model")
fig.colorbar(im, ax=ax, label="DSC")
plt.tight_layout()
plt.show()

## Summary

## Findings

**Pipeline status:** both baseline models run end-to-end on the 8-patient sanity
sample — inference → metrics → reporting, saved to
`results/segmentation/week5_baselines.csv`. UNet3D has been retired and removed from
the pipeline per the agreed scope reduction (no training from scratch, no fine-tuning
of external checkpoints).

- **nnU-Net (own cohort):** returns a zero mask until the locally trained
  `Dataset001_NSCLCRadiomics` checkpoint is available. The earlier all-zero result
  from the public Task006_Lung checkpoint was a v1/v2 format incompatibility, not a
  pipeline bug — that external-checkpoint path has been abandoned. The correct path is
  nnU-Net's self-configuring training on our own NSCLC-Radiomics GTV masks (see
  `notebooks/experiments/02_nnunet_vastai_training.ipynb`).

- **TotalSegmentator (`lung_nodules`):** unlike the earlier lung-lobe run, this uses
  the lesion-specific `lung_nodules` model (class 2 = lung nodule). It targets
  pulmonary nodules rather than the full GTV, so agreement with bulky tumour volumes
  is limited by design. See the summary table above for DSC / HD95 on the sanity
  sample; those numbers decide whether TotalSegmentator stands as a comparable
  segmenter or is repositioned as anatomical context (lung-region narrowing) for the
  Vision Agent.

- **Next steps:** run nnU-Net planning + training on the NSCLC-Radiomics cohort
  (compute plan stated up front) to replace the zero-mask fallback with a real
  tumour-segmentation baseline.